In [2]:
import pandas as pd
import pickle as pkl
import os
import json
from natsort import natsorted
import re
from tqdm.auto import tqdm
from ast import literal_eval
from collections import defaultdict


In [36]:
import spacy

# Load the SpaCy language model (replace 'en_core_web_sm' with a larger model if needed)
nlp = spacy.load('en_core_web_sm')


In [23]:
with open('hackerNews_updated.json', 'r') as f:
    file=json.load(f)

In [24]:
def remove_between_tags(text):
    # Remove everything between < and > including the symbols
    text = re.sub(r'<.*?>', '', text)
    # Remove any extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
for items in tqdm(file.keys()):
    content_list=file[items]['content']
    temp=[]
    for sentences in content_list:
        fixed=remove_between_tags(sentences)
        if fixed is not None:
            temp.append(remove_between_tags(sentences))
    file[items]['fixed_content']=temp

In [ ]:
file['30']['fixed_content']

In [ ]:
len(file)

In [19]:
results=os.listdir('outputs')

In [ ]:
results[0]

In [4]:
results=natsorted(results)

In [ ]:
results[0]

In [6]:
pkls_list=[]
for name in results:
    with open('outputs-old/{}'.format(name), 'rb') as f:
        pkls_list.append(pkl.load(f))

In [ ]:
pkls_list[0]

In [8]:
raw_json_str=pkls_list[0][0]

In [9]:
cleaned_json_str = raw_json_str.strip('```json\n').strip('\n```')

In [10]:
parsed_data = json.loads(cleaned_json_str)

In [ ]:
parsed_data

In [ ]:
len(pkls_list)

In [ ]:
pased_data_list=[]
for k,items in enumerate(pkls_list):
    try:
        raw_json_str=items[0]
        cleaned_json_str = raw_json_str.strip('```json\n').strip('\n```')
        parsed_data = json.loads(cleaned_json_str)
        pased_data_list.append(parsed_data)
    except:
        pased_data_list.append({})
        print(k)
        print(cleaned_json_str)

In [ ]:
pased_data_list[0]

In [ ]:
software=[]
hardware=[]
software_vulnerabilities=[]
hardware_vulnerabilities=[]
relations=[]
for items in pased_data_list:
    try:
        software.extend(items['Named_Entities']['software'])
        hardware.extend(items['Named_Entities']['hardware'])
        software_vulnerabilities.extend(items['Named_Entities']['software_vulnerabilities'])
        hardware_vulnerabilities.extend(items['Named_Entities']['hardware_vulnerabilities'])
        relations.extend(items['Relations'])
    except:
        try:
            print(items)
            software.extend(items['Named_Entities']['software'])
            hardware.extend(items['Named_Entities']['hardware'])
            software_vulnerabilities.extend(items['Named_Entities']['software_security_vulnerabilities'])
            hardware_vulnerabilities.extend(items['Named_Entities']['hardware_security_vulnerabilities'])
            relations.extend(items['Relations'])
        except:
            software.append([])
            hardware.append([])
            software_vulnerabilities.append([])
            hardware_vulnerabilities.append([])
            relations.append([])
            

In [ ]:
print(len(software))
print(len(hardware))
print(len(software_vulnerabilities))
print(len(hardware_vulnerabilities))
print(len(relations))


In [ ]:
software=list(set(software))
hardware=list(set(hardware))
software_vulnerabilities=list(set(software_vulnerabilities))
hardware_vulnerabilities=list(set(hardware_vulnerabilities))
relations=list(set(relations))

In [ ]:
print(len(software))
print(len(hardware))
print(len(software_vulnerabilities))
print(len(hardware_vulnerabilities))
print(len(relations))


In [75]:
software_per_article=[]
hardware_per_article=[]
software_vulnerabilities_per_article=[]
hardware_vulnerabilities_per_article=[]
relations_per_article=[]
for items in pased_data_list:
    try:
        software_per_article.append(items['Named_Entities']['software'])
        hardware_per_article.append(items['Named_Entities']['hardware'])
        software_vulnerabilities_per_article.append(items['Named_Entities']['software_vulnerabilities'])
        hardware_vulnerabilities_per_article.append(items['Named_Entities']['hardware_vulnerabilities'])
        relations_per_article.append(items['Relations'])
    except:
        # print(items)
        # try:
        #     software_per_article.append(items['Named_Entities']['software'])
        #     hardware_per_article.append(items['Named_Entities']['hardware'])
        #     software_vulnerabilities_per_article.append(items['Named_Entities']['software_security_vulnerabilities'])
        #     hardware_vulnerabilities_per_article.append(items['Named_Entities']['hardware_security_vulnerabilities'])
        #     relations_per_article.append(items['Relations'])
        # except:
            software_per_article.append([])
            hardware_per_article.append([])
            software_vulnerabilities_per_article.append([])
            hardware_vulnerabilities_per_article.append([])
            relations_per_article.append([])

In [ ]:
len(pased_data_list)

In [ ]:
software_per_article[234]

In [ ]:
len(hardware_vulnerabilities_per_article)

In [79]:
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_context(article, entity_name, window=3):
    """
    Extracts the surrounding context (window of sentences) around the named entity.
    :param article: str, full article text
    :param entity_name: str, the entity to find
    :param window: int, number of sentences before and after to include as context
    :return: list of extended context sentences
    """
    doc = nlp(article)
    sentences = list(doc.sents)
    contexts = []

    for i, sent in enumerate(sentences):
        if entity_name.lower() in sent.text.lower():
            start = max(0, i - window)
            end = min(len(sentences), i + window + 1)
            context = " ".join(sentences[start:end].text)
            contexts.append(context)

    return contexts

In [ ]:
text_file=[]
count=0
for items in tqdm(file.keys()):
    count+=1
    text_file.append(" ".join(file[items]['fixed_content']))

In [ ]:
text_file[0]

In [ ]:
software_entity_sentences=defaultdict(list)
for software, article in tqdm(zip(software_per_article, text_file)):
    for items in software:
        sent_list=extract_sentences_with_entity(article,items)
        if len(sent_list)!=0:
            for sent in sent_list:
                software_entity_sentences[items].append(sent)
    
    

In [ ]:
software_entity_sentences.keys()

In [ ]:
software_entity_sentences['Hunters International']

In [ ]:
hardware_entity_sentences=defaultdict(list)
for hardware, article in tqdm(zip(hardware_per_article, text_file)):
    for items in hardware:
        sent_list=extract_sentences_with_entity(article,items)
        if len(sent_list)!=0:
            for sent in sent_list:
                hardware_entity_sentences[items].append(sent)

In [ ]:
software_vulnerability_entity_sentences=defaultdict(list)
for software_vulnerability, article in tqdm(zip(software_vulnerability_per_article, text_file)):
    for items in software_vulnerability:
        sent_list=extract_sentences_with_entity(article,items)
        if len(sent_list)!=0:
            for sent in sent_list:
                software_vulnerability_entity_sentences[items].append(sent)

In [ ]:
hardware_vulnerability_entity_sentences=defaultdict(list)
for hardware_vulnerability, article in tqdm(zip(hardware_vulnerability_per_article, text_file)):
    for items in hardware_vulnerability:
        sent_list=extract_sentences_with_entity(article,items)
        if len(sent_list)!=0:
            for sent in sent_list:
                hardware_vulnerability_entity_sentences[items].append(sent)

In [90]:
with open('software_entity_sentences', 'wb') as f:
    pkl.dump(software_entity_sentences,f)

In [ ]:
with open('hardware_entity_sentences', 'wb') as f:
    pkl.dump(hardware_entity_sentences,f)

In [ ]:
with open('software_vulnerability_entity_sentences', 'wb') as f:
    pkl.dump(software_vulnerability_entity_sentences,f)

In [ ]:
with open('hardware_vulnerability_entity_sentences', 'wb') as f:
    pkl.dump(hardware_vulnerability_entity_sentences,f)

In [76]:
with open('hardware_vulnerability_entity_sentences', 'rb') as f:
    named_entities=pkl.load(f)

In [37]:
len(named_entities.keys())

3342

In [38]:
max_number=0
for k,v in named_entities.items():
    max_number=max(max_number, len(v))
    

In [ ]:
# from sentence_transformers import SentenceTransformer
# import numpy as np
# import hdbscan
# from sklearn.cluster import DBSCAN

# # # Sample named_entities dictionary
# # named_entities = {
# #     "Entity1": ["Sentence mentioning Entity1", "Another sentence for Entity1"],
# #     "Entity2": ["Sentence mentioning Entity2"],
# #     "Entity3": ["Entity3 appears in a different context"]
# # }

# # Flatten sentences and keep a mapping
# all_sentences = []
# sentence_to_entity = {}

# for entity, sentences in named_entities.items():
#     for sentence in sentences:
#         combined_text = f"{entity}: {sentence}"  # Combine entity name with sentence
#         all_sentences.append(combined_text)
#         sentence_to_entity[combined_text] = entity  # Map sentence+entity to original entity
        
# # Step 2: Convert Sentences to Embeddings
# model = SentenceTransformer('all-MiniLM-L6-v2')
# sentence_embeddings = model.encode(all_sentences)

# # Step 3: Cluster Embeddings
# dbscan = DBSCAN(eps=0.7, min_samples=1, metric='euclidean')  # Tuning eps may be needed
# cluster_labels = dbscan.fit_predict(sentence_embeddings)

# # Step 4: Assign Entity Clusters
# sentence_to_cluster = {sentence: label for sentence, label in zip(all_sentences, cluster_labels)}

# # Step 5: Map clusters to original entities
# cluster_to_entities = defaultdict(set)

# for sentence, cluster_label in sentence_to_cluster.items():
#     if cluster_label == -1:
#         cluster_label = f"Noise_{sentence_to_entity[sentence]}"  # Ignore noise points

#     entity = sentence_to_entity[sentence]
#     cluster_to_entities[cluster_label].add(entity)
    
#     # if cluster_label not in cluster_to_entities:
#     #     cluster_to_entities[cluster_label] = set()
    
#     cluster_to_entities[cluster_label].add(entity)

# # Step 6: Count Unique Resolved Entities
# unique_resolved_entities = len(cluster_to_entities)

# # Display Results
# print(f"Number of unique resolved entities: {unique_resolved_entities}")

# # Optional: Display cluster mappings
# for cluster_id, entities in cluster_to_entities.items():
#     print(f"Cluster {cluster_id}: {entities}")

/home/haskari/miniconda3/envs/influence/lib/python3.11/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Number of unique resolved entities: 2712
Cluster 0: {'CVE-2004-1464', 'CVE-2022-32893', 'Atlassian Confluence', 'CVE-2023-35182', 'CVE-2023-35359', 'CVE-2022-41091', 'CVE-2021-26897', 'CVE-2021-20023', 'CVE-2023-23397', 'CVE-2023-20872', 'phishing emails with malicious attachments', 'spear phishing', 'CVE-2021-40461', 'CVE-2019-5591', 'supply-chain attack', 'CVE-2021-41377', 'affiliate program', 'CVE-2021-34816', 'CVE-2023-34048', 'CVE-2022-38005', 'Supply chain attacks', 'CVE-2020-9934', 'CVE-2021-40487', 'CVE-2021-27422', 'CVE-2021-30533', 'CVE-2022-45444', 'CVE-2022-30222', 'CVE-2023-25610', 'CVE-2021-22899', 'CVE-2021-1048', 'CVE-2022-22039', 'malicious code', 'CVE-2023-31144', 'CVE-2022-1040', 'CVE-2022-40258', 'CVE-2023-40931', 'CVE-2022-22721', 'CVE-2016-7200', 'CVE-2022-25636', 'CVE-2021-3450', 'CVE-2023-38203', 'PWNYOURHOME', 'CVE-2018-11218', 'password spraying', 'ProxyShell', 'CVE-2023-35380', 'CVE-2022-22029', 'CVE-2023-30177', 'BEC', 'CVE-2021-33024', 'CVE-2022-20829', 'CV

In [ ]:
# import spacy
# import numpy as np
# from sentence_transformers import SentenceTransformer
# from sklearn.cluster import AgglomerativeClustering
# from collections import defaultdict

# # Load NLP model for sentence segmentation
# nlp = spacy.load("en_core_web_sm")

# # Load Sentence-BERT model
# sentence_bert_model = SentenceTransformer("all-MiniLM-L6-v2")

# def extract_context(article, entity_name, window=3):
#     """
#     Extracts a surrounding window of sentences around the named entity.
#     :param article: str, full article text
#     :param entity_name: str, the entity to find
#     :param window: int, number of sentences before and after to include as context
#     :return: list of extended context sentences
#     """
#     doc = nlp(article)
#     sentences = list(doc.sents)
#     contexts = []

#     for i, sent in enumerate(sentences):
#         if entity_name.lower() in sent.text.lower():
#             start = max(0, i - window)
#             end = min(len(sentences), i + window + 1)
#             context = " ".join(sentences[start:end].text)
#             contexts.append((context, entity_name))  # Keep track of the original entity

#     return contexts

# def get_sentence_embedding(text):
#     """
#     Compute embedding using Sentence-BERT.
#     :param text: str, input text
#     :return: np.array, embedding
#     """
#     return sentence_bert_model.encode(text)

# # Sample articles containing named entities
# articles = {
#     "Article1": """Apple announced a new iPhone. The company is also working on AI projects.
#                    Apple stock is rising. However, Apple is a fruit as well. 
#                    Many people eat apples every day.""",
    
#     "Article2": """Microsoft released Windows 11. Bill Gates founded Microsoft. 
#                    Microsoft Azure is growing in cloud computing. Meanwhile, 
#                    Microsoft is a well-known brand for software.""",

#     "Article3": """Orange is a popular fruit. Some people drink orange juice daily.
#                    However, Orange Telecom is a major company in France. 
#                    Orange is also a color used in design trends."""
# }

# # Step 1: Extract Sentences with Entities
# entity_sentences = []
# entity_labels = []

# for article_text in articles.values():
#     for entity in ["Apple", "Microsoft", "Orange"]:
#         extracted_contexts = extract_context(article_text, entity)
#         for context, entity_name in extracted_contexts:
#             entity_sentences.append(context)
#             entity_labels.append(entity_name)  # Store which entity it belongs to

# # Step 2: Compute Embeddings for Each Mention Separately
# sentence_embeddings = np.array([get_sentence_embedding(text) for text in entity_sentences])

# # Step 3: Cluster Entity Mentions (Separate Meanings First)
# clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=0.7).fit(sentence_embeddings)

# # Step 4: Group Clustered Mentions by Cluster ID
# cluster_to_entities = defaultdict(set)
# for i, cluster_label in enumerate(clustering.labels_):
#     cluster_to_entities[cluster_label].add(entity_labels[i])

# # Step 5: Count Unique Resolved Entities
# unique_resolved_entities = len(cluster_to_entities)

# # Display Results
# print(f"Number of unique resolved entities: {unique_resolved_entities}")

# # Display Clusters
# for cluster_id, entities in cluster_to_entities.items():
#     print(f"Cluster {cluster_id}: {entities}")


AttributeError: 'list' object has no attribute 'text'

In [29]:
length=0
for cluster_id, entities in cluster_to_entities.items():
    length+=len(entities)

In [30]:
length

282

In [ ]:
#Deepseek

In [71]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the SentenceBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Create a list of (entity, sentence) pairs
entity_sentence_pairs = []
for entity, sentences in named_entities.items():
    entity_sentence_pairs.extend([(entity, sentence) for sentence in sentences])

# Generate embeddings for all sentences
sentence_embeddings = model.encode([pair[1] for pair in entity_sentence_pairs], show_progress_bar=True)

# Create a dictionary to store embeddings for each entity
entity_embeddings = {}
for (entity, _), embedding in zip(entity_sentence_pairs, sentence_embeddings):
    if entity not in entity_embeddings:
        entity_embeddings[entity] = []
    entity_embeddings[entity].append(embedding)

# Average embeddings for each entity to get a single representation
entity_avg_embeddings = {entity: np.mean(embeddings, axis=0) for entity, embeddings in entity_embeddings.items()}

/home/haskari/miniconda3/envs/influence/lib/python3.11/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

In [72]:
from sklearn.cluster import DBSCAN

# Convert average embeddings to a matrix
embedding_matrix = np.array(list(entity_avg_embeddings.values()))

# Perform DBSCAN clustering
clustering = DBSCAN(eps=0.5, min_samples=1).fit(embedding_matrix)  # Adjust `eps` and `min_samples` as needed

# Assign cluster labels to entities
entity_clusters = {entity: label for entity, label in zip(entity_avg_embeddings.keys(), clustering.labels_)}

# Print clusters
for entity, cluster in entity_clusters.items():
    print(f"Entity: {entity}, Cluster: {cluster}")
    
    

Entity: RowHammer, Cluster: 0
Entity: RowPress, Cluster: 0
Entity: CVE-2023-45208, Cluster: 1
Entity: firmware backdoor, Cluster: 2
Entity: hardware supply chain attack, Cluster: 3
Entity: flaws in network-attached storage (NAS) devices, Cluster: 4
Entity: unpatched vulnerabilities affecting Baker Hughes' Bently Nevada 3500 rack model, Cluster: 5
Entity: GPU.zip, Cluster: 6
Entity: mobile device vulnerabilities, Cluster: 7
Entity: unpatched, Cluster: 8
Entity: unattended, Cluster: 9
Entity: misconfigured, Cluster: 9
Entity: security flaws, Cluster: 10
Entity: unverified ownership bug, Cluster: 11
Entity: flaws in the certified hardware, Cluster: 11
Entity: Collide+Power, Cluster: 12
Entity: CVE-2023-20583, Cluster: 13
Entity: Downfall, Cluster: 14
Entity: CVE-2022-40982, Cluster: 13
Entity: Inception, Cluster: 15
Entity: CVE-2023-20569, Cluster: 16
Entity: Zenbleed, Cluster: 13
Entity: CVE-2023-20593, Cluster: 13
Entity: Gather Data Sampling, Cluster: 17
Entity: Gather Value Injection,

In [73]:
from fuzzywuzzy import fuzz

# Example: Group entities with high string similarity
threshold = 80  # Adjust as needed
grouped_entities = {}
for entity in tqdm(entity_avg_embeddings.keys()):
    matched = False
    for grouped_entity in grouped_entities:
        if fuzz.ratio(entity.lower(), grouped_entity.lower()) > threshold:
            grouped_entities[grouped_entity].append(entity)
            matched = True
            break
    if not matched:
        grouped_entities[entity] = [entity]

# Print grouped entities
for key, values in grouped_entities.items():
    print(f"Group: {key}, Entities: {values}")

  0%|          | 0/238 [00:00<?, ?it/s]

Group: RowHammer, Entities: ['RowHammer', 'Rowhammer']
Group: RowPress, Entities: ['RowPress']
Group: CVE-2023-45208, Entities: ['CVE-2023-45208', 'CVE-2023-20583', 'CVE-2023-1620', 'CVE-2023-1748', 'CVE-2023-20078']
Group: firmware backdoor, Entities: ['firmware backdoor']
Group: hardware supply chain attack, Entities: ['hardware supply chain attack']
Group: flaws in network-attached storage (NAS) devices, Entities: ['flaws in network-attached storage (NAS) devices']
Group: unpatched vulnerabilities affecting Baker Hughes' Bently Nevada 3500 rack model, Entities: ["unpatched vulnerabilities affecting Baker Hughes' Bently Nevada 3500 rack model"]
Group: GPU.zip, Entities: ['GPU.zip']
Group: mobile device vulnerabilities, Entities: ['mobile device vulnerabilities']
Group: unpatched, Entities: ['unpatched']
Group: unattended, Entities: ['unattended']
Group: misconfigured, Entities: ['misconfigured']
Group: security flaws, Entities: ['security flaws', 'nine security flaws']
Group: unverif

In [50]:
from sklearn.cluster import AgglomerativeClustering

# Perform Agglomerative Clustering
clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=0.5).fit(embedding_matrix)

# Assign cluster labels to entities
entity_clusters = {entity: label for entity, label in zip(entity_avg_embeddings.keys(), clustering.labels_)}

# Print clusters
for entity, cluster in entity_clusters.items():
    print(f"Entity: {entity}, Cluster: {cluster}")

Entity: zero-day vulnerabilities, Cluster: 148
Entity: adversary-in-the-middle (AiTM) attacks, Cluster: 1947
Entity: CVE-2023-22515, Cluster: 147
Entity: CVE-2023-22518, Cluster: 33
Entity: watering hole attacks, Cluster: 2348
Entity: one-day exploits, Cluster: 1982
Entity: stolen credentials, Cluster: 2385
Entity: phishing, Cluster: 124
Entity: CVE-2023-47246, Cluster: 1566
Entity: unauthorized dissemination, Cluster: 2003
Entity: shadow IT, Cluster: 2129
Entity: unauthorized use, Cluster: 1897
Entity: CVE-2023-29552, Cluster: 2352
Entity: zero-click attacks, Cluster: 2324
Entity: spyware, Cluster: 23
Entity: file upload vulnerabilities, Cluster: 2239
Entity: malware, Cluster: 23
Entity: zero-day malware, Cluster: 281
Entity: Indicators of Compromise (IOCS), Cluster: 281
Entity: input validation testing, Cluster: 157
Entity: XSS testing, Cluster: 157
Entity: SQL injection testing, Cluster: 157
Entity: CVE-2023-38831, Cluster: 2402
Entity: CVE-2023-46604, Cluster: 125
Entity: CVE-2023-

In [59]:
with open('hardware_vulnerability_entity_sentences', 'rb') as f:
    named_entities=pkl.load(f)

In [74]:
#FINAL

model = SentenceTransformer('all-MiniLM-L6-v2')

# Create a list of (entity, sentence) pairs
entity_sentence_pairs = []
for entity, sentences in named_entities.items():
    entity_sentence_pairs.extend([(entity, sentence) for sentence in sentences])

# Generate embeddings for all sentences
sentence_embeddings = model.encode([pair[1] for pair in entity_sentence_pairs], show_progress_bar=True)

# Create a dictionary to store embeddings for each entity
entity_embeddings = {}
for (entity, sentence), embedding in zip(entity_sentence_pairs, sentence_embeddings):
    if entity not in entity_embeddings:
        entity_embeddings[entity] = []
    entity_embeddings[entity].append((sentence, embedding))

# # Step 4: Cluster Sentences for Each Entity
# def cluster_sentences_for_entity(embeddings, eps=0.9, min_samples=1):
#     """
#     Cluster sentences for a single entity using DBSCAN.
#     """
#     embedding_matrix = np.array([embedding for _, embedding in embeddings])
#     clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(embedding_matrix)
#     return clustering.labels_

def cluster_sentences_for_entity(embeddings, distance_threshold=0.9):
    """
    Dynamically select between Hierarchical Clustering and DBSCAN based on data size.
    """
    # if len(embeddings) == 1:
    #     return np.array([0])  # Assign single cluster if only one sentence

    # embedding_matrix = np.array([embedding for _, embedding in embeddings])

    # Use DBSCAN for small clusters (<5 sentences), Hierarchical for larger sets
    # if len(embedding_matrix) < 5:
    #     clustering = DBSCAN(eps=eps, min_samples=2).fit(embedding_matrix)
    # else:
    clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=distance_threshold).fit(embedding_matrix)

    return clustering.labels_

# Apply clustering to each entity
entity_clusters = {}
for entity, embeddings in tqdm(entity_embeddings.items()):
    labels = cluster_sentences_for_entity(embeddings, distance_threshold=0.5)
    entity_clusters[entity] = labels

# Step 5: Separate Entities by Context
# Group sentences into distinct contexts for each entity
entity_contexts = defaultdict(dict)
for entity, embeddings in entity_embeddings.items():
    labels = entity_clusters[entity]
    for (sentence, embedding), label in zip(embeddings, labels):
        if label not in entity_contexts[entity]:
            entity_contexts[entity][label] = []
        entity_contexts[entity][label].append((sentence, embedding))


# Step 6: Create Unique Representations for Each Context
# Average embeddings within each context to create a unique representation
context_embeddings = {}
for entity, contexts in tqdm(entity_contexts.items()):
    for label, sentences_embeddings in contexts.items():
        context_key = f"{entity}_context_{label}"  # Unique key for each context
        embeddings = np.array([embedding for _, embedding in sentences_embeddings])
        context_embeddings[context_key] = np.mean(embeddings, axis=0)

# Step 7: Cluster Across All Contexts
# Combine all context embeddings into a matrix
context_embedding_matrix = np.array(list(context_embeddings.values()))

# Perform DBSCAN clustering across all contexts
clustering = DBSCAN(eps=0.3, min_samples=2).fit(context_embedding_matrix)

# Assign cluster labels to each context
context_clusters = {context: label for context, label in zip(context_embeddings.keys(), clustering.labels_)}

# Step 8: Analyze Results
# Print clusters and their associated contexts
for context, cluster in context_clusters.items():
    print(f"Context: {context}, Cluster: {cluster}")

# Step 9: Group Entities by Final Clusters
# Create a dictionary to group entities by their final cluster labels
final_clusters = {}
for context, cluster in tqdm(context_clusters.items()):
    if cluster not in final_clusters:
        final_clusters[cluster] = []
    final_clusters[cluster].append(context)

# Print final clusters
print("\nFinal Clusters:")
for cluster, contexts in final_clusters.items():
    print(f"Cluster {cluster}: {contexts}")


/home/haskari/miniconda3/envs/influence/lib/python3.11/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/238 [00:00<?, ?it/s]

  0%|          | 0/238 [00:00<?, ?it/s]

Context: RowHammer_context_14, Cluster: 0
Context: RowPress_context_14, Cluster: 0
Context: CVE-2023-45208_context_14, Cluster: -1
Context: firmware backdoor_context_14, Cluster: -1
Context: hardware supply chain attack_context_14, Cluster: -1
Context: flaws in network-attached storage (NAS) devices_context_14, Cluster: -1
Context: unpatched vulnerabilities affecting Baker Hughes' Bently Nevada 3500 rack model_context_14, Cluster: -1
Context: GPU.zip_context_14, Cluster: -1
Context: mobile device vulnerabilities_context_14, Cluster: -1
Context: unpatched_context_14, Cluster: -1
Context: unattended_context_14, Cluster: -1
Context: misconfigured_context_14, Cluster: -1
Context: security flaws_context_14, Cluster: -1
Context: unverified ownership bug_context_14, Cluster: 1
Context: flaws in the certified hardware_context_14, Cluster: 1
Context: Collide+Power_context_14, Cluster: -1
Context: Collide+Power_context_121, Cluster: -1
Context: Collide+Power_context_180, Cluster: -1
Context: CVE

  0%|          | 0/302 [00:00<?, ?it/s]


Final Clusters:
Cluster 0: ['RowHammer_context_14', 'RowPress_context_14']
Cluster -1: ['CVE-2023-45208_context_14', 'firmware backdoor_context_14', 'hardware supply chain attack_context_14', 'flaws in network-attached storage (NAS) devices_context_14', "unpatched vulnerabilities affecting Baker Hughes' Bently Nevada 3500 rack model_context_14", 'GPU.zip_context_14', 'mobile device vulnerabilities_context_14', 'unpatched_context_14', 'unattended_context_14', 'misconfigured_context_14', 'security flaws_context_14', 'Collide+Power_context_14', 'Collide+Power_context_121', 'Collide+Power_context_180', 'Downfall_context_14', 'Downfall_context_121', 'Downfall_context_180', 'Downfall_context_150', 'Downfall_context_147', 'Inception_context_14', 'Inception_context_121', 'Inception_context_180', 'CVE-2023-20569_context_14', 'CVE-2023-20569_context_121', 'Zenbleed_context_14', 'Zenbleed_context_121', 'Zenbleed_context_180', 'CVE-2023-20593_context_14', 'Retbleed_context_14', 'Retbleed_context_

In [ ]:
from fuzzywuzzy import fuzz

def group_misspellings(final_clusters, threshold=80):
    """
    Group misspelled entities within final clusters using fuzzy string matching.
    :param final_clusters: Dictionary of final clusters (output from previous steps).
    :param threshold: Fuzzy matching threshold (0-100). Higher values mean stricter matching.
    :return: A dictionary with grouped misspellings.
    """
    # Flatten the final clusters into a list of all contexts
    all_contexts = [context for cluster in final_clusters.values() for context in cluster]

    # Create a dictionary to store grouped misspellings
    grouped_misspellings = {}

    # Iterate through all contexts and group similar ones
    for context in all_contexts:
        matched = False
        for group_key in grouped_misspellings:
            # Compare the current context with existing group keys using fuzzy matching
            if fuzz.ratio(context.lower(), group_key.lower()) > threshold:
                grouped_misspellings[group_key].append(context)
                matched = True
                break
        if not matched:
            grouped_misspellings[context] = [context]

    return grouped_misspellings

# Example usage
# final_clusters = {
#     0: ['Apple_context_0', 'Aple_context_0'],
#     1: ['Apple_context_1', 'apple_context_0'],
#     2: ['Banana_context_0', 'Bannana_context_0'],
# }

# Group misspellings using fuzzy matching
grouped_misspellings = group_misspellings(final_clusters, threshold=80)

# Print the grouped misspellings
print("Grouped Misspellings:")
for key, values in grouped_misspellings.items():
    print(f"Group: {key}, Entities: {values}")

In [ ]:
def merge_fuzzy_groups(final_clusters, grouped_misspellings):
    """
    Merge fuzzy groups back into the final clusters.
    :param final_clusters: Original final clusters.
    :param grouped_misspellings: Grouped misspellings from fuzzy matching.
    :return: Updated final clusters with merged groups.
    """
    # Create a mapping from old context to new group key
    context_to_group = {}
    for group_key, contexts in grouped_misspellings.items():
        for context in contexts:
            context_to_group[context] = group_key

    # Update final clusters with merged groups
    updated_clusters = {}
    for cluster_id, contexts in final_clusters.items():
        updated_clusters[cluster_id] = list(set([context_to_group[context] for context in contexts]))

    return updated_clusters

# Merge fuzzy groups back into final clusters
updated_clusters = merge_fuzzy_groups(final_clusters, grouped_misspellings)

# Print the updated clusters
print("\nUpdated Final Clusters:")
for cluster_id, contexts in updated_clusters.items():
    print(f"Cluster {cluster_id}: {contexts}")

In [95]:
with open('hardware_vulnerability_entity_sentences', 'rb') as f:
    named_entities=pkl.load(f)

In [99]:
import spacy
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import process, fuzz
from collections import defaultdict
from tqdm import tqdm

# Load NLP model for sentence segmentation
nlp = spacy.load("en_core_web_sm")

# Load Sentence-BERT model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Example Named Entities (with misspellings in entity names)


# Step 1: Fuzzy Matching to Correct Misspelled Named Entities
def fuzzy_match_entities(named_entities, threshold=65):
    """
    Corrects misspellings in named entity keys using fuzzy matching.
    If an entity name is similar (>threshold), merge their sentences.
    """
    corrected_entities = {}

    for entity in named_entities.keys():
        if not corrected_entities:  # If empty, directly add the first entity
            corrected_entities[entity] = named_entities[entity]
            continue

        # Find best fuzzy match among already corrected entities
        match_result = process.extractOne(entity, corrected_entities.keys(), scorer=fuzz.ratio)

        if match_result:  # Ensure match is found
            best_match, score, *_ = match_result  # Extract first two values, ignore others
            if score >= threshold:  # Merge with best-matching entity
                corrected_entities[best_match] += named_entities[entity]
            else:  # Keep as a new entity
                corrected_entities[entity] = named_entities[entity]
        else:  # No valid match found, add entity as is
            corrected_entities[entity] = named_entities[entity]

    return corrected_entities

# Apply fuzzy matching to entity names
named_entities = fuzzy_match_entities(named_entities)

# Step 2: Compute Sentence Embeddings
entity_embeddings = defaultdict(list)

for entity, sentences in named_entities.items():
    for sentence in sentences:
        embedding = model.encode(sentence)
        entity_embeddings[entity].append((sentence, embedding))

# Step 3: Convert Embeddings to Cosine Similarity Matrix
def cluster_sentences_using_cosine_similarity(embeddings, eps=0.3, min_samples=3):
    """
    Cluster sentences using DBSCAN with cosine similarity.
    """
    if len(embeddings) == 1:
        return np.array([0])  # Single cluster if only one sentence

    embedding_matrix = np.array([embedding for _, embedding in embeddings])
    
    # Compute cosine similarity
    similarity_matrix = cosine_similarity(embedding_matrix)
    
    # Convert similarity to distance (normalize to range [0,1])
    distance_matrix = 1 - similarity_matrix
    distance_matrix = np.clip(distance_matrix, 0, 1)  # Ensure valid range

    # Apply DBSCAN with precomputed distance matrix
    clustering = DBSCAN(eps=eps, min_samples=min_samples, metric="precomputed").fit(distance_matrix)
    return clustering.labels_

# Step 4: Apply Clustering to Each Entity
entity_clusters = {}
for entity, embeddings in tqdm(entity_embeddings.items()):
    labels = cluster_sentences_using_cosine_similarity(embeddings, eps=0.3, min_samples=3)
    entity_clusters[entity] = labels

# Step 5: Merge Over-Split Clusters Using Cosine Similarity
merged_entities = defaultdict(list)

for entity, embeddings in entity_embeddings.items():
    labels = entity_clusters[entity]
    unique_clusters = set(labels)

    # Compute mean embeddings for each cluster
    cluster_embeddings = {}
    for cluster in unique_clusters:
        cluster_embeddings[cluster] = np.mean(
            [embedding for (sentence, embedding), cluster_label in zip(embeddings, labels) if cluster_label == cluster],
            axis=0
        )

    # Compute pairwise cosine similarity between clusters
    cluster_keys = list(cluster_embeddings.keys())
    cluster_vectors = np.array([cluster_embeddings[key] for key in cluster_keys])
    cluster_similarities = cosine_similarity(cluster_vectors)

    # Merge clusters that are too similar (cosine similarity > 0.80)
    merged_labels = {}
    for i, cluster_i in enumerate(cluster_keys):
        for j, cluster_j in enumerate(cluster_keys):
            if i != j and cluster_similarities[i, j] > 0.75:
                merged_labels[cluster_j] = cluster_i  # Merge cluster_j into cluster_i

    # Assign final merged clusters
    for cluster, sentences in zip(labels, embeddings):
        final_label = merged_labels.get(cluster, cluster)
        new_entity_name = f"{entity} (Context {final_label})" if len(unique_clusters) > 1 else entity
        merged_entities[new_entity_name].append(sentences[0])

# Step 6: Display Results
print(f"\nNumber of unique resolved entities: {len(merged_entities)}\n")
for entity, sentences in merged_entities.items():
    print(f"Entity: {entity} - {len(sentences)} mentions")
    # for sentence in sentences:
    #     print(f"  - {sentence}")


/home/haskari/miniconda3/envs/influence/lib/python3.11/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
100%|██████████| 126/126 [00:00<00:00, 3104.88it/s]


Number of unique resolved entities: 135

Entity: RowHammer (Context -1) - 12 mentions
Entity: RowHammer (Context 1) - 10 mentions
Entity: RowPress - 1 mentions
Entity: CVE-2023-45208 (Context 3) - 43 mentions
Entity: CVE-2023-45208 (Context 0) - 4 mentions
Entity: CVE-2023-45208 (Context -1) - 26 mentions
Entity: CVE-2023-45208 (Context 2) - 3 mentions
Entity: firmware backdoor - 2 mentions
Entity: hardware supply chain attack - 1 mentions
Entity: flaws in network-attached storage (NAS) devices - 1 mentions
Entity: unpatched vulnerabilities affecting Baker Hughes' Bently Nevada 3500 rack model - 1 mentions
Entity: GPU.zip - 1 mentions
Entity: mobile device vulnerabilities - 15 mentions
Entity: unpatched - 3 mentions
Entity: unattended - 2 mentions
Entity: misconfigured - 1 mentions
Entity: security flaws - 2 mentions
Entity: unverified ownership bug - 1 mentions
Entity: flaws in the certified hardware - 1 mentions
Entity: Collide+Power - 4 mentions
Entity: Downfall - 6 mentions
Entity